# Fine-tuning Roberta on Jigsaw: Labels vs. Delta Regression

## Data loading and Constructing Delta

Load sample of Jigsaw data and ensure 'comment_text' is string type

In [1]:
from data.jigsaw import load_jigsaw_frame

JIGSAW_PATH = "../data/jigsaw/train.csv"
RANDOM_STATE = 42
N_SUB = 50_000

jf = load_jigsaw_frame(path=JIGSAW_PATH, sample=N_SUB, random_state=RANDOM_STATE)

# Ensure 'comment_text' is string type because there are some NaN values
jf = jf.dropna(subset=['comment_text']).copy()
mask_str = jf["comment_text"].apply(lambda x: isinstance(x, str))
jf = jf[mask_str].copy()
jf["comment_text"] = jf["comment_text"].astype(str)

print(jf.head())

                                              comment_text  n_yes  n_no  \
286892   What a breathe of fresh air to have someone wh...      1     5   
419218   Your jewish friends were the ones who told you...      6     4   
1055330  Possible collusion by Trump and his affiliates...      0     4   
1382764  Exactly.  We need a % of GDP spending cap at t...      0     4   
256049   By your own comment, even if some of them vote...      0     4   

         delta_signed  abs_delta  y_star  
286892      -1.098612   1.098612       0  
419218       0.336472   0.336472       1  
1055330     -1.609438   1.609438       0  
1382764     -1.609438   1.609438       0  
256049      -1.609438   1.609438       0  


## Create Training and Validation Split

In [2]:
from sklearn.model_selection import train_test_split

VAL_FRACTION = 0.1

train_jf, val_jf = train_test_split(
    jf,
    test_size=VAL_FRACTION,
    random_state=RANDOM_STATE,
    stratify=jf['y_star']
)

train_jf = train_jf.reset_index(drop=True)
val_jf = val_jf.reset_index(drop=True)

## Create Tokenizer and Pytorch Wrappers

In [3]:
from transformers import AutoTokenizer
from data.torch_datasets import JigsawTextDataset

MODEL_NAME = 'roberta-base'
MAX_LEN=128

# Tokenizer is used to convert text to input IDs and attention masks
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create pytorch wrappers for classification and regression tasks
train_dataset_cls = JigsawTextDataset(train_jf, tokenizer, max_len=MAX_LEN, task="classification")
val_dataset_cls   = JigsawTextDataset(val_jf,   tokenizer, max_len=MAX_LEN, task="classification")

train_dataset_reg = JigsawTextDataset(train_jf, tokenizer, max_len=MAX_LEN, task="regression")
val_dataset_reg   = JigsawTextDataset(val_jf,   tokenizer, max_len=MAX_LEN, task="regression")

# Inspect a sample from the classification dataset
sample = train_dataset_cls[0]
print(sample.keys())
print(sample["input_ids"].shape, sample["attention_mask"].shape)
print(sample["labels"], sample["weight"])



/storage/project/r-smussmann3-0/kkang68/projects/cost-sensitive-learning/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_keys(['input_ids', 'attention_mask', 'weight', 'labels'])
torch.Size([128]) torch.Size([128])
tensor(0) tensor(1.6094)


## Create Classification Model and Training Arguments

In [4]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

# Create model for sequence classification with 2 labels (toxic vs non-toxic)
model_cls = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="outputs/jigsaw_roberta_cls",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Train Classification Model

In [5]:
import numpy as np
from sklearn.metrics import accuracy_score
from transformers import Trainer

# Compute metrics function for classification using accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Create Trainer for classification task
trainer_cls = Trainer(
    model=model_cls,
    args=training_args,
    train_dataset=train_dataset_cls,
    eval_dataset=val_dataset_cls,
    compute_metrics=compute_metrics,
)

trainer_cls.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.174100,0.202555,0.945000
2,0.098900,0.192044,0.950800


TrainOutput(global_step=11250, training_loss=0.18878704092237683, metrics={'train_runtime': 441.4866, 'train_samples_per_second': 203.852, 'train_steps_per_second': 25.482, 'total_flos': 5919867190072320.0, 'train_loss': 0.18878704092237683, 'epoch': 2.0})

## Train Regression Model

In [6]:
model_reg = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
)
model_reg.config.problem_type = "regression"

training_args_reg = TrainingArguments(
    output_dir="outputs/jigsaw_roberta_reg",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
)

# Make sure that it uses squared error


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
def compute_metrics_reg(eval_pred):
    preds, delta_true = eval_pred      # delta_true comes from delta_signed
    delta_hat = preds.squeeze(-1)

    # Turn Δ and Δ̂ into labels via sign
    y_true = (delta_true >= 0).astype(int)
    y_pred = (delta_hat >= 0).astype(int)

    acc = accuracy_score(y_true, y_pred)
    return {"accuracy": acc}


trainer_reg = Trainer(
    model=model_reg,
    args=training_args_reg,
    train_dataset=train_dataset_reg,
    eval_dataset=val_dataset_reg,
    compute_metrics=compute_metrics_reg,
)

trainer_reg.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.196200,0.244276,0.946000
2,0.148900,0.185140,0.953600
3,0.116300,0.193318,0.953800
4,0.085500,0.198142,0.952200


TrainOutput(global_step=22500, training_loss=0.16459330339431763, metrics={'train_runtime': 883.181, 'train_samples_per_second': 203.804, 'train_steps_per_second': 25.476, 'total_flos': 1.1839628075947008e+16, 'train_loss': 0.16459330339431763, 'epoch': 4.0})

In [8]:
from sklearn.metrics import accuracy_score

eval_cls = trainer_cls.evaluate()
print("Classification eval:", eval_cls)

pred_output_cls = trainer_cls.predict(val_dataset_cls)
logits_cls = pred_output_cls.predictions
y_true = pred_output_cls.label_ids

y_pred_cls = np.argmax(logits_cls, axis=-1)
acc_cls = accuracy_score(y_true, y_pred_cls)

w = val_jf["abs_delta"].to_numpy()
correct = (y_true == y_pred_cls).astype(float)
weighted_acc_cls = (w * correct).sum() / w.sum()

print("Classification model:")
print("  Unweighted accuracy :", acc_cls)
print("  Weighted accuracy   :", weighted_acc_cls)


Classification eval: {'eval_loss': 0.19204428791999817, 'eval_accuracy': 0.9508, 'eval_runtime': 6.9205, 'eval_samples_per_second': 722.487, 'eval_steps_per_second': 90.311, 'epoch': 2.0}
Classification model:
  Unweighted accuracy : 0.9508
  Weighted accuracy   : 0.9819465577236803


In [9]:
pred_output_reg = trainer_reg.predict(val_dataset_reg)
logits_reg = pred_output_reg.predictions  # shape: (N, 1)
delta_hat  = logits_reg.squeeze(-1)       # shape: (N,)

# True labels for comparison (0/1)
y_true = val_jf["y_star"].to_numpy()
assert len(y_true) == len(delta_hat)

# Predicted labels from sign of δ̂
y_pred_reg = (delta_hat >= 0).astype(int)

In [10]:
w = val_jf["abs_delta"].to_numpy()
assert len(w) == len(y_true)

# Unweighted accuracy
acc_reg = accuracy_score(y_true, y_pred_reg)

# Weighted accuracy
correct = (y_true == y_pred_reg).astype(float)
weighted_acc_reg = (w * correct).sum() / w.sum()

print("Regression model:")
print("  Unweighted accuracy :", acc_reg)
print("  Weighted accuracy   :", weighted_acc_reg)

Regression model:
  Unweighted accuracy : 0.9538
  Weighted accuracy   : 0.9849406687540816


In [11]:
print("Classification model:  acc = 0.9498, weighted_acc = 0.9808")  # from before
print("Regression model:      acc = %.4f, weighted_acc = %.4f" % (acc_reg, weighted_acc_reg))

# Run 50_000 size subsamples, do 5 or 10 runs, get the average and standard error. Compute sample standard deviation by using (n-1); then, sample standard deviation divided by sqrt(n)



Classification model:  acc = 0.9498, weighted_acc = 0.9808
Regression model:      acc = 0.9538, weighted_acc = 0.9849
